# Benchmark: the Strands decider 2B on the same tests as Signpost

**Part 4 of "LLMs the Hard Way"** (branch `signpost`), next to `03_lora_signpost.ipynb`. [`StrandsAgents/strands-decider-2B-hobson-v19`](https://huggingface.co/StrandsAgents/strands-decider-2B-hobson-v19) is a ready-made **decision model**: Qwen3.5-2B-Base with a LoRA adapter and a small readout head. Given a *state* (text) and a question with named options, it returns a probability per option. Routing is exactly such a choice: the state is the conversation, and the options are the agents with their descriptions.

**The rule:** both models get **exactly the same test items**; only the presentation differs, since each model gets its own input format. The decider is used as published, zero-shot: nothing is trained here.

| Test | What it contains | Items |
|---|---|---|
| E10 | Realistic chats at 13 companies: route each user turn | 193 turns; 83 at the 6 test companies |
| E11 | One opening message, 5 agents | 100 |
| Banking | Banking conversations, 5 agents | 50 · 225 |
| E12 | Grounded questions, in the decider's native types: yes / no, choice, score | 170 |
| Independent | Written outside the project: questions, routing, first messages | 300 · 172 turns · 200 |
| Independent 2 | A second outside set: questions incl. 10–50 options, 8–15 agents per company | 200 · 70 turns · 50 |

Every question is timed as it runs (one at a time, warm, GPU synchronized, prompt building included), the same way as `03`. Cell 9 lines everything up against Signpost.

**Runs on:** a Kaggle GPU with Internet on. On a 2-GPU machine it uses the second GPU (`cuda:1`); set `DECIDER_DEVICE` to choose.

In [1]:
# Cell 1 — Setup: packages, device, progress log

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; the current peft (used by the decider) rejects it
!pip install -q "strands-decider" "transformers==5.17.0"

import json, os, subprocess, time
from importlib.metadata import version
from pathlib import Path

import torch

DECIDER_DEVICE = os.environ.get("DECIDER_DEVICE", "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0")
GPU = torch.device(DECIDER_DEVICE)


def log(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


log(f"torch {torch.__version__} | strands-decider {version('strands-decider')} | {torch.cuda.get_device_name(GPU)} | decider on {DECIDER_DEVICE}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 8.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.7/147.7 kB 3.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 9.9 MB/s eta 0:00:00:00:0100:01
[16:47:02] torch 2.11.0+cu128 | strands-decider 0.1.0 | Tesla T4 | decider on cuda:1


---
# Data and the decider

## Cell 2 — The tests

The same files as `03`, cloned (or refreshed) from this branch. E11 messages get their company's agents attached, so all first-message tests share one format.

In [2]:
# Cell 2 — All tests: E10, E11, banking, E12, and the two independent sets

DATA_BRANCH = "signpost"
repo = Path("llms-the-hard-way")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                    "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
DATA_DIR = repo / "data" / "modernbert_router"
load = lambda p: json.load(open(DATA_DIR / p))

real_convs = load("realistic/conversations.json")                                      # E10
ft = load("first_turn/messages.json")
e11 = [{**m, "agents": ft["companies"][m["company"]]} for m in ft["messages"]]          # E11
banking = load("banking/benchmark.json") + load("banking/extended.json")
e12 = load("choice_test/items.json")["items"]
ind_q, ind_r, ind_f = load("independent/questions.json")["items"], load("independent/routing.json"), load("independent/first_turn.json")["messages"]
ind2_q, ind2_r, ind2_f = (load("independent2/questions.json")["items"], load("independent2/routing.json"),
                          load("independent2/first_turn.json")["messages"])
BANKING_AGENTS = {                                          # exactly as in 02_frozen_router.ipynb, Cell 9, and 03
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, payment charges and "
                      "duplicate charges, refunds for transactions, and settlement problems where money was sent but "
                      "not received. Does not handle card servicing such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and unblocking, lost or "
                          "stolen cards, card replacement and delivery, card limits, card PIN, card security and "
                          "compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, investment transactions, "
                        "redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact details, issues that do "
                                      "not clearly belong to a specialist, and routing customers to the right agent.",
}
companies = sorted({c["company"] for c in real_convs})
TEST_COMPANIES = set(companies) - set(companies[::2])         # the 6 test companies of 02, Cell 16, and 03

turns = lambda convs: sum(len(c["user_turns"]) for c in convs)
print(f"E10 {len(real_convs)} conversations / {turns(real_convs)} turns | E11 {len(e11)} | banking {len(banking)} | E12 {len(e12)}")
print(f"independent: questions {len(ind_q)} | routing {turns(ind_r)} turns | first messages {len(ind_f)}")
print(f"independent 2: questions {len(ind2_q)} | routing {turns(ind2_r)} turns | first messages {len(ind2_f)}")

Cloning into 'llms-the-hard-way'...


E10 35 conversations / 193 turns | E11 100 | banking 225 | E12 170
independent: questions 300 | routing 172 turns | first messages 200
independent 2: questions 200 | routing 70 turns | first messages 50


## Cell 3 — Load the decider

`load_engine` downloads the base model (Qwen3.5-2B-Base) with the checkpoint's adapter and head. It runs in the checkpoint's own **bfloat16**; a T4 has no fast bfloat16 path and emulates it, so the decider's latency here is an upper bound. Every decider number in the ledger was measured this way. A model-card question checks that it answers sensibly.

In [3]:
# Cell 3 — Load the decider, and ask the model card's example question

from strands_decider.infer import load_engine
from strands_decider.schema import ChoiceQuestion, NoulQuestion, ScoreQuestion

CHECKPOINT = "StrandsAgents/strands-decider-2B-hobson-v19"
t0 = time.perf_counter()
engine = load_engine(CHECKPOINT, device=DECIDER_DEVICE)
log(f"loaded {CHECKPOINT} in {time.perf_counter() - t0:.0f}s | dtype {next(engine.model.parameters()).dtype}")

demo = engine.ask("Help! My payouts have been failing for 3 days!",
                  {"team": ChoiceQuestion(instructions="Which team should handle this?",
                                          criteria={"billing": "billing", "sales": "sales", "retail": "retail"})})
print("model card example ->", demo.answers["team"].choice, demo.answers["team"].probabilities)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 37 files:   0%|          | 0/37 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[16:47:56] loaded StrandsAgents/strands-decider-2B-hobson-v19 in 36s | dtype torch.bfloat16


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


model card example -> billing {'billing': 0.7634, 'sales': 0.1213, 'retail': 0.1154}


## Cell 4 — How each test is asked

- **Routing:** the state is the conversation as a plain transcript, one line per message, with the latest user message repeated at the end; the question is *"Which agent should handle the user's latest message?"*; the options are the agents with their descriptions. `window` keeps only the last N user turns. The decider has **no current-agent input**, so every turn is routed on its own.
- **Questions**, in the decider's native types: yes / no → `NoulQuestion` (P(true) > 0.5 → "yes"); choice and many-option → `ChoiceQuestion`; score → `ScoreQuestion` with the levels low to high (the most likely level; its expected level is kept too).
- `timed` measures every call; `table` prints right / total per group, as in `03`.

In [4]:
# Cell 4 — The routing question, the native question types, timing and a table helper

ROUTE_Q = "Which agent should handle the user's latest message?"
TURN_TYPES = ["start", "answer", "follow", "aspect", "closing", "also_stay", "switch", "short_switch", "return", "handoff_ok", "branch"]
TYPE = {"noul": "yes/no", "choice": "choice", "score": "score", "many_choice": "many-option"}
R, LAT = {}, {}                                       # every result as (right, total); every test's latencies (ms)


def state_text(messages, window=None):
    """The transcript up to the latest user message, optionally only the last `window` user turns."""
    if window is not None:
        user_idx = [i for i, m in enumerate(messages) if m["role"] == "user"]
        messages = messages[user_idx[-window]:] if len(user_idx) > window else messages
    return "Conversation:\n" + "\n".join(f"{m['role']}: {m['content']}" for m in messages) + f"\n\nLatest user message: {messages[-1]['content']}"


def decide(messages, agents, window=None):
    """-> (chosen agent, {agent: probability}, most likely first)."""
    ans = engine.ask(state_text(messages, window), {"route": ChoiceQuestion(instructions=ROUTE_Q, criteria=dict(agents))}).answers["route"]
    return ans.choice, dict(sorted(ans.probabilities.items(), key=lambda kv: -kv[1]))


def ask_item(it):
    """A grounded question in the decider's native type -> (pick, expected level for score questions or None)."""
    if it["type"] == "noul":
        p_true = engine.ask(it["state"], {"q": NoulQuestion(instructions=it["question"])}).answers["q"].noul
        return ("yes" if p_true > 0.5 else "no"), None
    if it["type"] in ("choice", "many_choice"):
        return engine.ask(it["state"], {"q": ChoiceQuestion(instructions=it["question"], criteria={o: o for o in it["options"]})}).answers["q"].choice, None
    ans = engine.ask(it["state"], {"q": ScoreQuestion(instructions=it["question"], criteria=it["options"])}).answers["q"]
    probs = [ans.probabilities.get(o, ans.probabilities.get(str(i), 0.0)) for i, o in enumerate(it["options"])]   # keyed by name or index
    top = max(range(len(probs)), key=probs.__getitem__) if any(probs) else min(len(it["options"]) - 1, max(0, round(ans.score)))
    return it["options"][top], float(ans.score)


def timed(fn):
    torch.cuda.synchronize(GPU)
    t0 = time.perf_counter()
    out = fn()
    torch.cuda.synchronize(GPU)
    return out, (time.perf_counter() - t0) * 1000


def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "n": len(ms)}


def table(rows, key, cols, groups=None):
    """Print right / total per group (key(row) -> group) and for ALL rows; cols = {column: row -> True if right}."""
    groups = list(groups or sorted({key(r) for r in rows})) + ["ALL"]
    w = max(len(str(g_)) for g_ in groups)
    print(f"  {'':{w}s} | {'n':>4} | " + " | ".join(f"{c:>12s}" for c in cols))
    for g_ in groups:
        rs = [r for r in rows if g_ == "ALL" or key(r) == g_]
        if rs:
            print(f"  {g_:{w}s} | {len(rs):>4} | " + " | ".join(f"{sum(map(f_, rs)):>5} / {len(rs):<4}" for f_ in cols.values()))


def n_ok(rows, key="ok"):
    return sum(r[key] for r in rows), len(rows)


print(state_text(real_convs[0]["messages"][:5]))
print(decide(real_convs[0]["messages"][:5], real_convs[0]["agents"]))

Conversation:
user: Hi, I need a refill on my blood pressure tablets.
assistant: Of course. Is it the amlodipine from Dr Shah?
user: yes that one
assistant: Done, it'll be ready tomorrow after 2pm.
user: Can I take it with ibuprofen? I've got a bad back.

Latest user message: Can I take it with ibuprofen? I've got a bad back.
('Pharmacist Advice Agent', {'Pharmacist Advice Agent': 0.8201, 'Repeat Prescriptions Agent': 0.1728, 'Home Delivery Agent': 0.0071})


---
# Tests

## Cell 5 — Conversations: every user turn

E10 and both independent sets, with the **full history** (what Signpost sees) and with a **window** of the last 3 user turns, per turn type. E10 is also reported for the 6 test companies, as in `02` and `03`.

In [5]:
# Cell 5 — Conversations: every user turn routed with the history so far (full and window 3)

def route_conversations(convs, window=None):
    rows, lat = [], []
    for conv in convs:
        for u in conv["user_turns"]:
            (pick, _), ms = timed(lambda: decide(conv["messages"][:u["message_index"] + 1], conv["agents"], window))
            rows.append({"company": conv.get("company"), "type": u["turn_type"], "ok": pick == u["expected_agent"]})
            lat.append(ms)
    return rows, lat


decide([{"role": "user", "content": "warm-up"}], BANKING_AGENTS)                      # warm-up (not timed)
for name, convs in [("E10", real_convs), ("IND", ind_r), ("IND2", ind2_r)]:
    log(f"{name}: routing every turn, full history and window 3")
    full, LAT[f"{name} routing"] = route_conversations(convs)
    win, LAT[f"{name} routing, window 3"] = route_conversations(convs, window=3)
    R[f"{name} routing"], R[f"{name} routing, window 3"] = n_ok(full), n_ok(win)
    both = [{"company": a["company"], "type": a["type"], "full": a["ok"], "win": b["ok"]} for a, b in zip(full, win)]
    cols = {"full history": lambda r: r["full"], "window 3": lambda r: r["win"]}
    table(both, lambda r: r["type"], cols, TURN_TYPES)
    if name == "E10":
        test = [r for r in both if r["company"] in TEST_COMPANIES]
        R["E10 test companies"], R["E10 test companies, window 3"] = n_ok(test, "full"), n_ok(test, "win")
        print(f"E10 at the 6 test companies ({', '.join(sorted(TEST_COMPANIES))}):")
        table(test, lambda r: r["type"], cols, TURN_TYPES)

[16:47:59] E10: routing every turn, full history and window 3
               |    n | full history |     window 3
  start        |   35 |    33 / 35   |    33 / 35  
  answer       |   51 |    41 / 51   |    47 / 51  
  follow       |   21 |    18 / 21   |    21 / 21  
  aspect       |    7 |     7 / 7    |     7 / 7   
  closing      |   15 |    11 / 15   |    12 / 15  
  also_stay    |    4 |     4 / 4    |     4 / 4   
  switch       |   42 |    26 / 42   |    30 / 42  
  short_switch |    6 |     2 / 6    |     3 / 6   
  return       |    7 |     6 / 7    |     3 / 7   
  handoff_ok   |    2 |     1 / 2    |     1 / 2   
  branch       |    3 |     2 / 3    |     2 / 3   
  ALL          |  193 |   151 / 193  |   163 / 193 
E10 at the 6 test companies (cruise line, driving school, games console company, pet food subscription, photo printing service, solar installer):
               |    n | full history |     window 3
  start        |   15 |    14 / 15   |    14 / 15  
  answer    

## Cell 6 — First messages

One message, no history: E11 and both independent sets, top-1 and top-2 per message type, and E11's mistakes.

In [6]:
# Cell 6 — First messages: E11, independent, independent 2

def first_messages(msgs):
    rows, lat = [], []
    for m in msgs:
        (pick, probs), ms = timed(lambda: decide([{"role": "user", "content": m["message"]}], m["agents"]))
        rows.append({**m, "pick": pick, "p": probs[pick], "ok": pick == m["target_agent"], "top2": m["target_agent"] in list(probs)[:2]})
        lat.append(ms)
    return rows, lat


for name, msgs in [("E11", e11), ("IND", ind_f), ("IND2", ind2_f)]:
    log(f"{name}: first messages")
    rows, LAT[f"{name} first"] = first_messages(msgs)
    R[f"{name} first top-1"], R[f"{name} first top-2"] = n_ok(rows), n_ok(rows, "top2")
    table(rows, lambda r: r["type"], {"top-1": lambda r: r["ok"], "top-2": lambda r: r["top2"]})
    if name == "E11":
        for r in [r for r in rows if not r["ok"]]:
            print(f"    {r['type']:8s} | {r['message'][:58]:58s} | expected {r['target_agent'].replace(' Agent', ''):24s} | "
                  f"got {r['pick'].replace(' Agent', '')} ({r['p']:.2f})")

[16:53:26] E11: first messages
           |    n |        top-1 |        top-2
  clear    |   31 |    31 / 31   |    31 / 31  
  close    |   20 |    17 / 20   |    20 / 20  
  indirect |   17 |    16 / 17   |    17 / 17  
  noisy    |   15 |    15 / 15   |    15 / 15  
  short    |   17 |    17 / 17   |    17 / 17  
  ALL      |  100 |    96 / 100  |   100 / 100 
    close    | my garden waste bin was skipped this week                  | expected Garden Waste             | got Bin Collections (0.89)
    indirect | The courier didn't turn up for the pickup I booked.        | expected Shipping Labels          | got Disputes (0.69)
    close    | I booked the wrong day by mistake, can I swap to Friday?   | expected Refunds & Exchanges      | got Ticket Booking (0.77)
    close    | How do I give my accountant access without seeing payroll? | expected Users & Login            | got Payroll (0.69)
[16:53:56] IND: first messages
           |    n |        top-1 |        top-2
  clear    |  

## Cell 7 — Banking

The banking conversations of `02` and `03`, last user message routed. The decider's training data includes banking77 and CLINC-OOS, so this set isn't unseen for it.

In [7]:
# Cell 7 — Banking: 225 conversations, the first 50 are the benchmark

log("banking")
rows, LAT["banking"] = [], []
for ex in banking:
    (pick, _), ms = timed(lambda: decide(ex["messages"], BANKING_AGENTS))
    rows.append({"ok": pick == ex["target_agent"]})
    LAT["banking"].append(ms)
R["banking 50"], R["banking 225"] = n_ok(rows[:50]), n_ok(rows)
print(f"banking: benchmark {R['banking 50'][0]} / 50 | all {R['banking 225'][0]} / {len(rows)}")

[16:55:17] banking
banking: benchmark 48 / 50 | all 214 / 225


## Cell 8 — Grounded questions

The decider on its home ground: E12 and both independent sets in its native question types. E12 also shows per category, the question groups (same text, different questions) and every mistake; the independent set shows its flagged items separately (22 claim-check yes / no items, all answering "yes").

In [8]:
# Cell 8 — Questions: E12, independent, independent 2

def questions(items):
    rows, lat = [], []
    for it in items:
        (pick, expected), ms = timed(lambda: ask_item(it))
        rows.append({**it, "pick": pick, "ok": pick == it["answer"], "expected_level": expected,
                     "level_error": abs(it["options"].index(pick) - it["answer_index"]) if it["type"] == "score" else None})
        lat.append(ms)
    return rows, lat


ask_item(e12[0])                                              # warm-up (not timed)
for name, items in [("E12", e12), ("IND", ind_q), ("IND2", ind2_q)]:
    log(f"{name}: questions")
    rows, LAT[f"{name} questions"] = questions(items)
    R[f"{name} questions"] = n_ok(rows)
    R.update({f"{name} {TYPE[t]}": n_ok([r for r in rows if r["type"] == t]) for t in TYPE if any(r["type"] == t for r in rows)})
    table(rows, lambda r: TYPE[r["type"]], {"right": lambda r: r["ok"]}, [TYPE[t] for t in TYPE])
    sc = [r for r in rows if r["type"] == "score"]
    print(f"  score within 1 level: {sum(r['level_error'] <= 1 for r in sc)} / {len(sc)}")
    if name == "E12":
        table(rows, lambda r: r["category"], {"right": lambda r: r["ok"]})
        groups = {}
        for r in rows:
            if "pair" in r:
                groups.setdefault(r["pair"], []).append(r)
        print(f"  question groups all right: {sum(all(r['ok'] for r in g) for g in groups.values())} / {len(groups)}")
        for r in [r for r in rows if not r["ok"]]:
            print(f"    {r['id']} {r['type']:6s} | {r['state'][:44]:44s} | {r['question'][:36]:36s} | expected {r['answer']:14s} | got {r['pick']}")
    if name == "IND":
        R["IND questions unflagged"] = n_ok([r for r in rows if not r.get("flag")])
        claim = [r for r in rows if r.get("flag") == "claim_check"]
        print(f"  without flagged items {R['IND questions unflagged'][0]} / {R['IND questions unflagged'][1]} | "
              f"claim-check yes/no {n_ok(claim)[0]} / {len(claim)}")

[16:57:07] E12: questions
              |    n |        right
  yes/no      |   58 |    56 / 58  
  choice      |   59 |    56 / 59  
  score       |   53 |    48 / 53  
  ALL         |  170 |   160 / 170 
  score within 1 level: 53 / 53
           |    n |        right
  compare  |   12 |    12 / 12  
  detail   |   38 |    33 / 38  
  fact     |   37 |    36 / 37  
  intent   |   22 |    21 / 22  
  language |    2 |     2 / 2   
  routing  |   14 |    13 / 14  
  tone     |   28 |    28 / 28  
  topic    |    4 |     4 / 4   
  urgency  |   13 |    11 / 13  
  ALL      |  170 |   160 / 170 
  question groups all right: 10 / 11
    n22 noul   | Can I change the delivery address to my offi | Is the customer asking to change the | expected no             | got yes
    n23 noul   | The parcel was left with my neighbour at num | Was the parcel delivered?            | expected yes            | got no
    c09 choice | I can't do Thursday any more; let's keep Tue | Which day does the writer

---
# Results

## Cell 9 — Side by side with Signpost

Signpost is **not run in this notebook**: its columns are results measured elsewhere on the same items. "Run 25 (published)" is the model on the Hugging Face Hub, tested in its own run (independent set 2: the Hub model, tested in `05_listwise_signpost.ipynb`); "03 final run" is the retrained, unpublished model from `03_lora_signpost.ipynb`. Signpost routes conversations live with its own previous pick as the current agent; the decider routes each turn on its own, shown with the full history and with a 3-turn window. Latency is p50 in ms on a T4, one question at a time.

In [9]:
# Cell 9 — The decider next to Signpost (run 25 = published; 03 final run), then latency

SIGNPOST = {
    "run 25 (published)": {"E10 routing": 181, "E10 test companies": 78, "IND routing": 133, "IND2 routing": 62, "E11 first top-1": 97,
                           "IND first top-1": 180, "IND2 first top-1": 47, "banking 50": 48, "banking 225": 210,
                           "E12 questions": 161, "E12 yes/no": 58, "E12 choice": 58, "E12 score": 45,
                           "IND questions": 221, "IND yes/no": 75, "IND choice": 103, "IND score": 43,
                           "IND2 questions": 188, "IND2 yes/no": 47, "IND2 choice": 47, "IND2 score": 45, "IND2 many-option": 49},
    "03 final run": {"E10 routing": 181, "E10 test companies": 80, "IND routing": 135, "IND2 routing": 63, "E11 first top-1": 96,
                     "E11 first top-2": 100, "IND first top-1": 178, "IND first top-2": 193, "IND2 first top-1": 50, "IND2 first top-2": 50,
                     "banking 50": 49, "banking 225": 216, "E12 questions": 159, "E12 yes/no": 55, "E12 choice": 56, "E12 score": 48,
                     "IND questions": 219, "IND yes/no": 73, "IND choice": 104, "IND score": 42, "IND questions unflagged": 219,
                     "IND2 questions": 190, "IND2 yes/no": 47, "IND2 choice": 47, "IND2 score": 47, "IND2 many-option": 49},
}
SIGNPOST_P50 = {"E10 routing": 58, "IND routing": 70, "IND2 routing": 187, "E11 first": 57, "IND first": 57, "IND2 first": 73,
                "banking": 73, "E12 questions": 55, "IND questions": 55, "IND2 questions": 57}     # 03 final run

print(f"{'result':30s} | {'decider':>11} | " + " | ".join(f"{'Signpost ' + c:>29s}" for c in SIGNPOST))
for k, (ok_, n) in R.items():
    print(f"{k:30s} | {f'{ok_} / {n}':>11} | " + " | ".join(f"{SIGNPOST[c].get(k, '—'):>29}" for c in SIGNPOST))

print(f"\n{'latency (ms)':24s} | {'decider p50 / p95':>18} | {'Signpost p50':>12}")
for k, v in LAT.items():
    L = lat_summary(v)
    print(f"{k:24s} | {L['p50']:>8.0f} / {L['p95']:<7.0f} | {SIGNPOST_P50.get(k, '—'):>12}")

result                         |     decider |   Signpost run 25 (published) |         Signpost 03 final run
E10 routing                    |   151 / 193 |                           181 |                           181
E10 routing, window 3          |   163 / 193 |                             — |                             —
E10 test companies             |     62 / 83 |                            78 |                            80
E10 test companies, window 3   |     69 / 83 |                             — |                             —
IND routing                    |   113 / 172 |                           133 |                           135
IND routing, window 3          |   121 / 172 |                             — |                             —
IND2 routing                   |     55 / 70 |                            62 |                            63
IND2 routing, window 3         |     59 / 70 |                             — |                             —
E11 first top-1    

---
## Cell 10 — What we learned about the decider

From this run (Cell 9). The decider is deterministic: every number on E10, E11, banking, E12 and the independent set reproduces the ledger's benchmark B1 (`experiments/modernbert_router_ledger.md`, runs 14, 19 and 22) exactly. Signpost's side is in `03`: the published model (run 25) and the notebook's final run.

**Where it is strong**
- **Grounded questions:** E12 160 / 170 and independent 253 / 300, against Signpost's 159–161 and 219–221. The gap is almost all **score** (independent 70 / 92 against Signpost's 42–43): it's a general reader trained on about 30 public classification datasets. On independent set 2 it is level instead (185 / 200 against 188–190), and its score there is 43 / 50 against Signpost's 45–47.
- **First messages:** E11 96 / 100 (top-2 100), independent 178 / 200, independent 2 50 / 50: level with Signpost (97 · 180 · 47 for run 25; 96 · 178 · 50 for 03's final run).

**Where it is weak**
- **Switches inside a conversation.** It has no current-agent input, so every turn is routed blind. E10 real switches: 10 / 20 at the test companies (13 with a window). Independent: short switches 2 / 20 (6 with a window) and switches 18 / 46 (21); independent 2: switches 5 / 11 (6) and closings 2 / 7 (5). These cases were written outside the project, so this isn't an artefact of our own test writing. Overall live routing: E10 151 (163 with a window) against Signpost's 181; independent 113 (121) against 133–135; independent 2 55 (59) against 62–63.
- **History length.** With the full history, older topics pull its answer back: E10 151 / 193. A window of the last 3 user turns lifts it to 163 (answers +6, switches +4), but returns to an earlier topic get worse (6 → 3 of 7; independent 2: 5 → 3 of 5) because the topic falls out of the window. We tried the same window on our router (`02`, run 10b): no gain, because we pass the current agent.
- **Double negation.** On the independent set's 22 "does the text contradict the claim …" items it answered 0 / 22 (as did our model). These are reported separately.

**Caveats on the comparison**
- **Banking isn't unseen for it:** banking77 and CLINC-OOS are in its training data (48 · 214 / 225, against Signpost's 48–49 · 210–216).
- **Its latency is an upper bound.** It ran in bfloat16 on a T4, which has no fast bfloat16 path, and without its optional kernels (`flash-linear-attention`, `causal-conv1d` fell back to plain PyTorch). Even so: p50 about 200 ms per question and 290–650 ms per routing decision (growing with the history and the number of agents), against Signpost's 55–73 ms (187 ms with up to 11 agents). Costs also scale differently: the decider's with prompt length (history + all options in one prompt), Signpost's with the number of options (one pass each, batched).
- **It's zero-shot and general-purpose;** Signpost was trained for exactly these question types, on data written by the same author as most of our tests. The independent sets are the fairest comparison.

---
## Next

**Part 5** (branch `signpost-apps`) uses Signpost as the router of a real multi-agent app: LangGraph with five Claude agents, a DynamoDB checkpointer and a message reducer.